In [2]:
!pip -q install requests beautifulsoup4 pandas rapidfuzz playwright


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.9 MB/s eta 0:00:00


In [5]:
from __future__ import annotations

import re
import json
import os
import csv
import gzip
import time
import html as html_lib
import subprocess
import sys
import base64
from pathlib import Path
from datetime import datetime, timedelta, date
from typing import Dict, List, Optional, Tuple
from urllib.parse import urljoin, quote, urlparse, parse_qs, unquote

import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag
from rapidfuzz import fuzz
from IPython.display import display, HTML


# =========================
# CONFIG
# =========================
AMC_BASE = "https://www.amctheatres.com"
RT_BASE  = "https://www.rottentomatoes.com"

THEATRES = [
    {"name": "AMC Tustin 14 @ The District",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-tustin-14-at-the-district/showtimes"},
    {"name": "AMC Orange 30",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-orange-30/showtimes"},
    {"name": "AMC Woodbridge 5",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-woodbridge-5/showtimes"},
]

# Optional: pin a specific weekend Saturday (YYYY-MM-DD) for testing
OVERRIDE_SATURDAY = None  # e.g. "2025-12-20"

# Turn on to see RT URL attempts
DEBUG_RT = False

IMDB_DATASET_DIR = Path("build/imdb_datasets")
IMDB_BASICS_GZ = IMDB_DATASET_DIR / "title.basics.tsv.gz"
IMDB_RATINGS_GZ = IMDB_DATASET_DIR / "title.ratings.tsv.gz"
IMDB_BASICS_URL = "https://datasets.imdbws.com/title.basics.tsv.gz"
IMDB_RATINGS_URL = "https://datasets.imdbws.com/title.ratings.tsv.gz"


def safe_html_attr(value: object) -> str:
    return html_lib.escape(str(value or ''), quote=True)


def link_html(url: Optional[str], text: object) -> str:
    label = '' if text is None else str(text)
    if isinstance(url, str) and url.startswith('http'):
        return f'<a href="{safe_html_attr(url)}" target="_blank" rel="noopener noreferrer">{html_lib.escape(label)}</a>'
    return html_lib.escape(label)


def normalize_title_for_match(title: str) -> str:
    s = (title or '').lower().strip()
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    s = re.sub(r'\b(the|a|an)\b', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def _clean_missing_text(value: object) -> str:
    s = '' if value is None else str(value).strip()
    return '' if s.lower() in {'', 'none', 'nan', 'null', '\n'} else s


def _safe_int(value: object) -> Optional[int]:
    s = _clean_missing_text(value)
    if not s:
        return None
    try:
        return int(s)
    except Exception:
        return None


def _safe_float(value: object) -> Optional[float]:
    s = _clean_missing_text(value)
    if not s:
        return None
    try:
        return float(s)
    except Exception:
        return None


def _download_to_path(session: requests.Session, url: str, dest: Path, tries: int = 3) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + '.part')
    last_err = None
    for attempt in range(tries):
        try:
            with session.get(url, stream=True, timeout=180, allow_redirects=True) as r:
                r.raise_for_status()
                with tmp.open('wb') as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            tmp.replace(dest)
            return dest
        except Exception as e:
            last_err = e
            try:
                tmp.unlink(missing_ok=True)
            except Exception:
                pass
            time.sleep(attempt + 1)
    raise RuntimeError(f'Failed downloading {url}: {last_err}')


def search_result_urls(session: requests.Session, query: str, allowed_prefixes: tuple[str, ...], max_results: int = 8) -> List[str]:
    urls: List[str] = []
    seen = set()
    endpoints = [
        ('https://html.duckduckgo.com/html/', {'q': query}),
        ('https://duckduckgo.com/html/', {'q': query}),
    ]
    for endpoint, params in endpoints:
        try:
            r = session.get(endpoint, params=params, timeout=30, allow_redirects=True)
            r.raise_for_status()
        except Exception:
            continue

        soup = BeautifulSoup(r.text or '', 'html.parser')
        for a in soup.find_all('a', href=True):
            href = (a.get('href') or '').strip()
            if not href:
                continue
            resolved = href
            if 'duckduckgo.com/l/?' in href or href.startswith('/l/?'):
                parsed = urlparse(href)
                qs = parse_qs(parsed.query)
                resolved = unquote((qs.get('uddg') or [''])[0])
            if not resolved.startswith('http'):
                continue
            if not any(resolved.startswith(p) for p in allowed_prefixes):
                continue
            clean = resolved.split('?', 1)[0]
            if clean not in seen:
                urls.append(clean)
                seen.add(clean)
            if len(urls) >= max_results:
                return urls
    return urls


def build_imdb_lookup(session: requests.Session, titles: List[str]) -> Dict[str, Tuple[Optional[float], Optional[int], Optional[str]]]:
    title_variants: Dict[str, List[str]] = {}
    variant_to_titles: Dict[str, set[str]] = {}
    for title in titles:
        norms: List[str] = []
        seen = set()
        for variant in candidate_title_variants(title):
            norm = normalize_title_for_match(variant)
            if norm and norm not in seen:
                norms.append(norm)
                seen.add(norm)
                variant_to_titles.setdefault(norm, set()).add(title)
        title_variants[title] = norms

    _download_to_path(session, IMDB_BASICS_URL, IMDB_BASICS_GZ)
    _download_to_path(session, IMDB_RATINGS_URL, IMDB_RATINGS_GZ)

    candidate_map: Dict[str, List[dict]] = {title: [] for title in titles}
    with gzip.open(IMDB_BASICS_GZ, 'rt', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            if row.get('titleType') not in {'movie', 'tvMovie'}:
                continue
            if row.get('isAdult') == '1':
                continue

            primary_norm = normalize_title_for_match(row.get('primaryTitle', ''))
            original_norm = normalize_title_for_match(row.get('originalTitle', ''))
            matched_titles = set()
            if primary_norm:
                matched_titles |= variant_to_titles.get(primary_norm, set())
            if original_norm:
                matched_titles |= variant_to_titles.get(original_norm, set())
            if not matched_titles:
                continue

            cand = {
                'tconst': row.get('tconst', ''),
                'titleType': row.get('titleType', ''),
                'primaryTitle': row.get('primaryTitle', ''),
                'originalTitle': row.get('originalTitle', ''),
                'primaryNorm': primary_norm,
                'originalNorm': original_norm,
                'startYear': _safe_int(row.get('startYear')),
                'runtimeMinutes': _safe_int(row.get('runtimeMinutes')),
            }
            for title in matched_titles:
                candidate_map[title].append(cand)

    if not any(candidate_map.values()):
        return {title: (None, None, None) for title in titles}

    rating_rows: Dict[str, Tuple[Optional[float], int]] = {}
    wanted_ids = {cand['tconst'] for cands in candidate_map.values() for cand in cands if cand.get('tconst')}
    with gzip.open(IMDB_RATINGS_GZ, 'rt', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            tconst = row.get('tconst', '')
            if tconst not in wanted_ids:
                continue
            rating_rows[tconst] = (_safe_float(row.get('averageRating')), _safe_int(row.get('numVotes')) or 0)

    current_year = date.today().year
    out: Dict[str, Tuple[Optional[float], Optional[int], Optional[str]]] = {}
    for title in titles:
        target = normalize_title_for_match(title)
        chosen = None
        chosen_key = None
        for cand in candidate_map.get(title, []):
            rating_val, votes = rating_rows.get(cand['tconst'], (None, 0))
            exact = int(target in {cand['primaryNorm'], cand['originalNorm']})
            fuzz_score = max(
                fuzz.token_set_ratio(cand['primaryNorm'], target) if cand['primaryNorm'] else 0,
                fuzz.token_set_ratio(cand['originalNorm'], target) if cand['originalNorm'] else 0,
            )
            year = cand.get('startYear')
            recent_bias = 0
            if year is not None:
                if year >= current_year - 2:
                    recent_bias = 2
                elif year >= current_year - 10:
                    recent_bias = 1
            score_key = (
                exact,
                fuzz_score,
                1 if cand.get('titleType') == 'movie' else 0,
                recent_bias,
                votes,
                year or 0,
            )
            if chosen is None or score_key > chosen_key:
                chosen = (rating_val, cand.get('runtimeMinutes'), f"https://www.imdb.com/title/{cand['tconst']}/")
                chosen_key = score_key

        if chosen_key is not None and (chosen_key[0] == 1 or chosen_key[1] >= 90):
            out[title] = chosen
        else:
            out[title] = (None, None, None)
    return out


# =========================
# REGEX / CONSTANTS
# =========================
SHOWTIME_HREF_RE = re.compile(
    r"""(?:https?://(?:www\.)?amctheatres\.com)?/showtimes(?:/all/[^\s"'?#]+/[^\s"'?#]+/[^\s"'?#]+)?/\d+""",
    re.I,
)
MOVIE_HREF_RE    = re.compile(r"(?:https?://(?:www\.)?amctheatres\.com)?/movies/", re.I)
TIME_RE          = re.compile(r"(\d{1,2}:\d{2}\s*[ap]m)", re.I)
AMC_RUNTIME_RE   = re.compile(r"(\d+)\s*hr?\s*(\d+)\s*min|(\d+)\s*min", re.I)

# Text fallbacks (handle normal % and fullwidth ％)
RT_TOMA_TXT_1 = re.compile(r"(\d{1,3})\s*[％%]\s*Tomatometer", re.I)
RT_TOMA_TXT_2 = re.compile(r"Tomatometer\s*(\d{1,3})\s*[％%]", re.I)

RT_AUD_TXT_1  = re.compile(r"(\d{1,3})\s*[％%]\s*(Popcornmeter|Audience\s*Score)", re.I)
RT_AUD_TXT_2  = re.compile(r"(Popcornmeter|Audience\s*Score)\s*(\d{1,3})\s*[％%]", re.I)

# JSON-ish fallbacks
RT_TOMA_JSON_1 = re.compile(r'"tomatometerScore"\s*:\s*(\d{1,3})', re.I)
RT_TOMA_JSON_2 = re.compile(r'"tomatometerScore"\s*:\s*\{[^}]{0,300}?"value"\s*:\s*(\d{1,3})', re.I)

RT_AUD_JSON_1  = re.compile(r'"audienceScore"\s*:\s*(\d{1,3})', re.I)
RT_AUD_JSON_2  = re.compile(r'"audienceScore"\s*:\s*\{[^}]{0,300}?"value"\s*:\s*(\d{1,3})', re.I)
RT_AUD_JSON_3  = re.compile(r'"popcornmeter"\s*:\s*(\d{1,3})', re.I)
RT_AUD_JSON_4  = re.compile(r'"popcornmeter"\s*:\s*\{[^}]{0,300}?"score"\s*:\s*(\d{1,3})', re.I)


# =========================
# TIME / SESSION HELPERS
# =========================
def upcoming_weekend_pacific() -> Tuple[date, date]:
    """Return upcoming Saturday/Sunday in America/Los_Angeles."""
    try:
        from zoneinfo import ZoneInfo
        today = datetime.now(ZoneInfo("America/Los_Angeles")).date()
    except Exception:
        today = date.today()

    if OVERRIDE_SATURDAY:
        sat = datetime.strptime(OVERRIDE_SATURDAY, "%Y-%m-%d").date()
        return sat, sat + timedelta(days=1)

    wd = today.weekday()  # Mon=0 .. Sun=6
    sat = today + timedelta(days=(5 - wd)) if wd <= 5 else today + timedelta(days=6)
    return sat, sat + timedelta(days=1)

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://www.google.com/",
    })
    return s

def fetch_html(session: requests.Session, url: str, params: dict | None = None, tries: int = 3) -> Tuple[int, str]:
    """Return (status_code, text). Retries a few times."""
    last_status, last_text = 0, ""
    for i in range(tries):
        try:
            r = session.get(url, params=params, timeout=30, allow_redirects=True)
            last_status = r.status_code
            last_text = r.text or ""
            if last_status == 200 and "Access Denied" not in last_text:
                return last_status, last_text
        except Exception:
            pass
        time.sleep(1.0 * (i + 1))
    return last_status, last_text



def looks_like_amc_interstitial(html_txt: str, final_url: str = "") -> bool:
    low = (html_txt or "").lower()
    final_low = (final_url or "").lower()
    markers = (
        "the site requires javascript to be enabled",
        "queue.amctheatres.com",
        "global safety net",
        "enable-javascript.com",
        "access denied",
        "verify you are human",
        "checking your browser before accessing",
        "attention required",
        "cf-chl",
        "captcha",
        "bot protection",
    )
    return any(m in low for m in markers) or "queue.amctheatres.com" in final_low


AMC_QUEUE_REDIRECT_RE = re.compile(
    r"""document\.location\.href\s*=\s*decodeURIComponent\(\s*['"]([^'"]+)['"]\s*\)""",
    re.I,
)


def extract_amc_interstitial_redirect(html_txt: str, current_url: str = AMC_BASE) -> Optional[str]:
    """
    AMC's queue/interstitial page embeds a one-time redirect target in inline JS like:
      document.location.href = decodeURIComponent('%2F%3Fc%3Damctheatres...')
    Following that URL is the important step; simply reloading the original showtimes URL
    often just returns another interstitial.
    """
    if not html_txt:
        return None

    m = AMC_QUEUE_REDIRECT_RE.search(html_txt)
    if not m:
        return None

    raw = m.group(1)
    try:
        decoded = unquote(raw)
    except Exception:
        decoded = raw

    return urljoin(current_url or AMC_BASE, decoded)


def fetch_html(session: requests.Session, url: str, params: dict | None = None, tries: int = 3) -> Tuple[int, str]:
    """Return (status_code, text). Retries a few times and follows AMC queue redirect tokens when present."""
    last_status, last_text = 0, ""
    full_url = requests.Request("GET", url, params=params).prepare().url

    for i in range(tries):
        try:
            # A harmless cookie that mirrors AMC's own cookie-support check.
            session.cookies.set("cookietest", "1", domain="www.amctheatres.com", path="/")

            r = session.get(url, params=params, timeout=30, allow_redirects=True)
            last_status = r.status_code
            last_text = r.text or ""

            if last_status == 200 and "Access Denied" not in last_text and not looks_like_amc_interstitial(last_text, r.url):
                return last_status, last_text

            # AMC queue pages include an encoded redirect target. Follow it explicitly.
            redirect_url = extract_amc_interstitial_redirect(last_text, r.url or full_url)
            if redirect_url:
                r2 = session.get(redirect_url, timeout=30, allow_redirects=True)
                last_status = r2.status_code
                last_text = r2.text or ""
                if last_status == 200 and "Access Denied" not in last_text and not looks_like_amc_interstitial(last_text, r2.url):
                    return last_status, last_text
        except Exception:
            pass

        time.sleep(1.0 * (i + 1))

    return last_status, last_text



def fetch_html_with_browser(url: str, params: dict | None = None, timeout_ms: int = 30000) -> Tuple[int, str]:
    """
    Use a real browser for AMC pages, but run Playwright in a subprocess so Jupyter's
    asyncio loop does not break sync_playwright(). This keeps the last-known-working
    parser logic from v5 while avoiding the notebook event-loop warning.
    """
    full_url = requests.Request("GET", url, params=params).prepare().url
    query_string = urlparse(full_url).query
    browser_fetch_script = base64.b64decode("CmltcG9ydCBqc29uCmltcG9ydCByZQppbXBvcnQgc3lzCmZyb20gdXJsbGliLnBhcnNlIGltcG9ydCB1bnF1b3RlLCB1cmxqb2luCmZyb20gcGxheXdyaWdodC5zeW5jX2FwaSBpbXBvcnQgc3luY19wbGF5d3JpZ2h0CgpBTUNfQkFTRSA9ICJodHRwczovL3d3dy5hbWN0aGVhdHJlcy5jb20iClFVRVVFX1JFID0gcmUuY29tcGlsZShyJycnZG9jdW1lbnRcLmxvY2F0aW9uXC5ocmVmXHMqPVxzKmRlY29kZVVSSUNvbXBvbmVudFwoXHMqWyciXShbXiciXSspWyciXVxzKlwpJycnLCByZS5JKQpNQVJLRVJTID0gKAogICAgInRoZSBzaXRlIHJlcXVpcmVzIGphdmFzY3JpcHQgdG8gYmUgZW5hYmxlZCIsCiAgICAicXVldWUuYW1jdGhlYXRyZXMuY29tIiwKICAgICJnbG9iYWwgc2FmZXR5IG5ldCIsCiAgICAiZW5hYmxlLWphdmFzY3JpcHQuY29tIiwKICAgICJhY2Nlc3MgZGVuaWVkIiwKICAgICJ2ZXJpZnkgeW91IGFyZSBodW1hbiIsCiAgICAiY2hlY2tpbmcgeW91ciBicm93c2VyIGJlZm9yZSBhY2Nlc3NpbmciLAogICAgImF0dGVudGlvbiByZXF1aXJlZCIsCiAgICAiY2YtY2hsIiwKICAgICJjYXB0Y2hhIiwKICAgICJib3QgcHJvdGVjdGlvbiIsCikKCmRlZiBsb29rc19saWtlX2ludGVyc3RpdGlhbChodG1sX3R4dDogc3RyLCBmaW5hbF91cmw6IHN0ciA9ICIiKSAtPiBib29sOgogICAgbG93ID0gKGh0bWxfdHh0IG9yICIiKS5sb3dlcigpCiAgICBmaW5hbF9sb3cgPSAoZmluYWxfdXJsIG9yICIiKS5sb3dlcigpCiAgICByZXR1cm4gYW55KG0gaW4gbG93IGZvciBtIGluIE1BUktFUlMpIG9yICJxdWV1ZS5hbWN0aGVhdHJlcy5jb20iIGluIGZpbmFsX2xvdwoKZGVmIGV4dHJhY3RfcmVkaXJlY3QoaHRtbF90eHQ6IHN0ciwgY3VycmVudF91cmw6IHN0ciA9IEFNQ19CQVNFKToKICAgIGlmIG5vdCBodG1sX3R4dDoKICAgICAgICByZXR1cm4gTm9uZQogICAgbSA9IFFVRVVFX1JFLnNlYXJjaChodG1sX3R4dCkKICAgIGlmIG5vdCBtOgogICAgICAgIHJldHVybiBOb25lCiAgICByYXcgPSBtLmdyb3VwKDEpCiAgICB0cnk6CiAgICAgICAgZGVjb2RlZCA9IHVucXVvdGUocmF3KQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBkZWNvZGVkID0gcmF3CiAgICByZXR1cm4gdXJsam9pbihjdXJyZW50X3VybCBvciBBTUNfQkFTRSwgZGVjb2RlZCkKCmJhc2VfdXJsID0gc3lzLmFyZ3ZbMV0KcXMgPSBzeXMuYXJndlsyXQp0aW1lb3V0X21zID0gaW50KHN5cy5hcmd2WzNdKQp0YXJnZXRfdXJsID0gYmFzZV91cmwgKyAoKCI/IiArIHFzKSBpZiBxcyBlbHNlICIiKQoKd2l0aCBzeW5jX3BsYXl3cmlnaHQoKSBhcyBwOgogICAgYnJvd3NlciA9IHAuY2hyb21pdW0ubGF1bmNoKAogICAgICAgIGhlYWRsZXNzPVRydWUsCiAgICAgICAgYXJncz1bCiAgICAgICAgICAgICItLWRpc2FibGUtYmxpbmstZmVhdHVyZXM9QXV0b21hdGlvbkNvbnRyb2xsZWQiLAogICAgICAgICAgICAiLS1uby1zYW5kYm94IiwKICAgICAgICBdLAogICAgKQogICAgY29udGV4dCA9IGJyb3dzZXIubmV3X2NvbnRleHQoCiAgICAgICAgdXNlcl9hZ2VudD0oCiAgICAgICAgICAgICJNb3ppbGxhLzUuMCAoWDExOyBMaW51eCB4ODZfNjQpIEFwcGxlV2ViS2l0LzUzNy4zNiAiCiAgICAgICAgICAgICIoS0hUTUwsIGxpa2UgR2Vja28pIENocm9tZS8xMjMuMC4wLjAgU2FmYXJpLzUzNy4zNiIKICAgICAgICApLAogICAgICAgIGxvY2FsZT0iZW4tVVMiLAogICAgICAgIHRpbWV6b25lX2lkPSJBbWVyaWNhL0xvc19BbmdlbGVzIiwKICAgICAgICB2aWV3cG9ydD17IndpZHRoIjogMTQ0MCwgImhlaWdodCI6IDIyMDB9LAogICAgICAgIGphdmFfc2NyaXB0X2VuYWJsZWQ9VHJ1ZSwKICAgICAgICBleHRyYV9odHRwX2hlYWRlcnM9eyJBY2NlcHQtTGFuZ3VhZ2UiOiAiZW4tVVMsZW47cT0wLjkifSwKICAgICkKICAgIGNvbnRleHQuYWRkX2luaXRfc2NyaXB0KAogICAgICAgICcnJwogICAgICAgIE9iamVjdC5kZWZpbmVQcm9wZXJ0eShuYXZpZ2F0b3IsICJ3ZWJkcml2ZXIiLCB7IGdldDogKCkgPT4gdW5kZWZpbmVkIH0pOwogICAgICAgIHdpbmRvdy5jaHJvbWUgPSB3aW5kb3cuY2hyb21lIHx8IHsgcnVudGltZToge30gfTsKICAgICAgICBPYmplY3QuZGVmaW5lUHJvcGVydHkobmF2aWdhdG9yLCAibGFuZ3VhZ2VzIiwgeyBnZXQ6ICgpID0+IFsiZW4tVVMiLCAiZW4iXSB9KTsKICAgICAgICBPYmplY3QuZGVmaW5lUHJvcGVydHkobmF2aWdhdG9yLCAicGx1Z2lucyIsIHsgZ2V0OiAoKSA9PiBbMSwgMiwgMywgNF0gfSk7CiAgICAgICAgJycnCiAgICApCiAgICBjb250ZXh0LmFkZF9jb29raWVzKFt7CiAgICAgICAgIm5hbWUiOiAiY29va2lldGVzdCIsCiAgICAgICAgInZhbHVlIjogIjEiLAogICAgICAgICJkb21haW4iOiAid3d3LmFtY3RoZWF0cmVzLmNvbSIsCiAgICAgICAgInBhdGgiOiAiLyIsCiAgICB9XSkKCiAgICBwYWdlID0gY29udGV4dC5uZXdfcGFnZSgpCiAgICByZXNwb25zZSA9IHBhZ2UuZ290byh0YXJnZXRfdXJsLCB3YWl0X3VudGlsPSJkb21jb250ZW50bG9hZGVkIiwgdGltZW91dD10aW1lb3V0X21zKQogICAgc3RhdHVzID0gcmVzcG9uc2Uuc3RhdHVzIGlmIHJlc3BvbnNlIGlzIG5vdCBOb25lIGVsc2UgMjAwCiAgICBwYWdlLndhaXRfZm9yX3RpbWVvdXQoMTUwMCkKICAgIGh0bWxfdHh0ID0gcGFnZS5jb250ZW50KCkKCiAgICBmb3IgXyBpbiByYW5nZSgzKToKICAgICAgICBpZiBub3QgbG9va3NfbGlrZV9pbnRlcnN0aXRpYWwoaHRtbF90eHQsIHBhZ2UudXJsKToKICAgICAgICAgICAgYnJlYWsKCiAgICAgICAgcmVkaXJlY3RfdXJsID0gZXh0cmFjdF9yZWRpcmVjdChodG1sX3R4dCwgcGFnZS51cmwgb3IgdGFyZ2V0X3VybCkKICAgICAgICB0cnk6CiAgICAgICAgICAgIGlmIHJlZGlyZWN0X3VybDoKICAgICAgICAgICAgICAgIHJlc3BvbnNlID0gcGFnZS5nb3RvKHJlZGlyZWN0X3VybCwgd2FpdF91bnRpbD0iZG9tY29udGVudGxvYWRlZCIsIHRpbWVvdXQ9dGltZW91dF9tcykKICAgICAgICAgICAgICAgIHN0YXR1cyA9IHJlc3BvbnNlLnN0YXR1cyBpZiByZXNwb25zZSBpcyBub3QgTm9uZSBlbHNlIHN0YXR1cwogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcGFnZS5yZWxvYWQod2FpdF91bnRpbD0iZG9tY29udGVudGxvYWRlZCIsIHRpbWVvdXQ9dGltZW91dF9tcykKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXNzCgogICAgICAgIHRyeToKICAgICAgICAgICAgcGFnZS53YWl0X2Zvcl9sb2FkX3N0YXRlKCJuZXR3b3JraWRsZSIsIHRpbWVvdXQ9NTAwMCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXNzCiAgICAgICAgcGFnZS53YWl0X2Zvcl90aW1lb3V0KDIwMDApCiAgICAgICAgaHRtbF90eHQgPSBwYWdlLmNvbnRlbnQoKQoKICAgIHBheWxvYWQgPSB7CiAgICAgICAgInN0YXR1cyI6IGludChzdGF0dXMgb3IgMjAwKSwKICAgICAgICAiaHRtbCI6IHBhZ2UuY29udGVudCgpLAogICAgICAgICJ1cmwiOiBwYWdlLnVybCwKICAgIH0KCiAgICBjb250ZXh0LmNsb3NlKCkKICAgIGJyb3dzZXIuY2xvc2UoKQoKcHJpbnQoanNvbi5kdW1wcyhwYXlsb2FkKSkK").decode()

    proc = subprocess.run(
        [sys.executable, "-c", browser_fetch_script, url, query_string, str(timeout_ms)],
        capture_output=True,
        text=True,
        timeout=max(90, int(timeout_ms / 1000) * 4),
    )
    if proc.returncode != 0:
        raise RuntimeError((proc.stderr or proc.stdout or "browser subprocess failed").strip())

    try:
        payload = json.loads(proc.stdout)
    except Exception as e:
        raise RuntimeError(f"browser subprocess returned invalid JSON: {proc.stdout[:500]}") from e

    return int(payload.get("status") or 200), str(payload.get("html") or "")


def fetch_amc_html(session: requests.Session, url: str, params: dict | None = None) -> Tuple[int, str]:
    status, html_txt = fetch_html(session, url, params=params, tries=2)
    if status == 200 and not looks_like_amc_interstitial(html_txt):
        return status, html_txt

    target = requests.Request("GET", url, params=params).prepare().url
    print(f"[INFO] AMC returned a JS/queue page for {target}; retrying with Playwright browser fallback...")

    try:
        status2, html_txt2 = fetch_html_with_browser(url, params=params)
        if status2 == 200 and not looks_like_amc_interstitial(html_txt2):
            return status2, html_txt2
        return status2, html_txt2
    except Exception as e:
        print(f"[WARN] Browser fallback failed for {target}: {e}")
        return status, html_txt


# =========================
# AMC SCRAPING
# =========================
def extract_time(txt: str) -> Optional[str]:
    m = TIME_RE.search(txt or "")
    if not m:
        return None
    return re.sub(r"\s+", " ", m.group(1).strip().lower())

def parse_time_for_sort(t: str) -> int:
    try:
        dt = datetime.strptime(t.strip().upper(), "%I:%M %p")
        return dt.hour * 60 + dt.minute
    except Exception:
        return 10**9

def is_a_list_excluded_near_tag(tag: Tag, max_parent_levels: int = 6) -> bool:
    """
    More reliable AMC parsing: only look within a limited number of ancestor levels
    so we don't accidentally match page-wide legends/footers.
    """
    lvl = 0
    for p in tag.parents:
        if not isinstance(p, Tag):
            continue
        txt = p.get_text(" ", strip=True).lower()
        if "excluded from a-list" in txt:
            return True
        lvl += 1
        if lvl >= max_parent_levels:
            break
    return False

def normalize_format_label(txt: str) -> Optional[str]:
    t = _normalize_space(txt)
    if not t:
        return None
    tl = t.lower()

    if "open caption" in tl:
        return "Open Caption (On-screen Subtitles)"
    if "closed caption" in tl:
        return "Closed Caption"
    if "spoken with spanish subtitles" in tl or "english spoken with spanish subtitles" in tl:
        return "English Spoken with Spanish Subtitles"
    if "laser" in tl:
        return "Laser at AMC"
    if "dolby" in tl:
        return "Dolby Cinema at AMC"
    if "imax" in tl:
        return "IMAX"
    if "prime" in tl:
        return "PRIME at AMC"
    if "reald" in tl:
        return "RealD 3D"
    if "fan faves" in tl:
        return "Fan Faves"
    if "no trailers" in tl:
        return "No Trailers"
    return t

def looks_like_format_label(txt: str) -> bool:
    t = _normalize_space(txt)
    if not t:
        return False
    tl = t.lower()
    if len(t) > 90 or len(t.split()) > 10:
        return False
    bad_substrings = [
        "amc orange 30", "amc tustin 14", "amc woodbridge 5",
        "showtimes", "movie theater", "get tickets", "buy tickets",
        "available at", "excluded from a-list"
    ]
    if any(b in tl for b in bad_substrings):
        return False
    needles = [
        "imax", "dolby", "prime", "reald", "laser", "fan faves",
        "no trailers", "spoken", "subtitles", "open caption", "closed caption"
    ]
    return any(n in tl for n in needles)

def _strings_one_level(x: object) -> List[str]:
    out: List[str] = []
    if isinstance(x, str):
        out.append(x)
    elif isinstance(x, dict):
        for v in x.values():
            if isinstance(v, str):
                out.append(v)
            elif isinstance(v, list):
                for item in v:
                    if isinstance(item, str):
                        out.append(item)
                    elif isinstance(item, dict):
                        for vv in item.values():
                            if isinstance(vv, str):
                                out.append(vv)
    elif isinstance(x, list):
        for item in x:
            if isinstance(item, str):
                out.append(item)
            elif isinstance(item, dict):
                for vv in item.values():
                    if isinstance(vv, str):
                        out.append(vv)
    return out


def href_matches_movie(href: str) -> bool:
    return bool(MOVIE_HREF_RE.search((href or "").strip()))

def href_matches_showtime(href: str) -> bool:
    return bool(SHOWTIME_HREF_RE.search((href or "").strip()))

def _nearest_movie_container(tag: Tag) -> Tag:
    for p in [tag, *tag.parents]:
        if not isinstance(p, Tag):
            continue
        if p.name in {"article", "section", "li"}:
            return p
        classes = " ".join(p.get("class", [])).lower()
        if any(k in classes for k in ["movie", "showtimes", "grid", "card", "tile", "session"]):
            return p
    return tag


def _normalize_space(txt: str) -> str:
    return re.sub(r"\s+", " ", (txt or "")).strip()

def looks_like_title_text(txt: str) -> bool:
    t = _normalize_space(txt)
    if not t or len(t) < 2 or len(t) > 160:
        return False
    tl = t.lower()
    if TIME_RE.search(t) or AMC_RUNTIME_RE.search(t):
        return False
    bad_substrings = [
        "showtimes", "get tickets", "buy tickets", "read more", "find showtimes",
        "amc stubs", "excluded from a-list", "reserved seating", "details",
        "watch trailer", "trailer", "movie theater", "available at", "fan faves",
        "closed caption", "open caption", "spoken subtitles", "subtitle",
        "imax", "dolby", "laser at amc", "prime at amc", "reald", "ticket"
    ]
    if any(b in tl for b in bad_substrings):
        return False
    if re.search(r"^(sat|sun|mon|tue|wed|thu|fri)\b", tl):
        return False
    if re.search(r"^(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\b", tl):
        return False
    if re.search(r"^[\d\W_]+$", t):
        return False
    return any(ch.isalpha() for ch in t)

def _container_has_showtime_or_runtime(tag: Tag) -> bool:
    try:
        container = _nearest_movie_container(tag)
        txt = _normalize_space(container.get_text(" ", strip=True))
        return bool(TIME_RE.search(txt) or AMC_RUNTIME_RE.search(txt))
    except Exception:
        return False

def _candidate_title_tags(soup: BeautifulSoup) -> List[Tag]:
    out: List[Tag] = []
    seen = set()

    for el in soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"]):
        txt = _normalize_space(el.get_text(" ", strip=True))
        if looks_like_title_text(txt) and _container_has_showtime_or_runtime(el):
            key = (txt.lower(), getattr(el, "sourceline", None))
            if key not in seen:
                out.append(el)
                seen.add(key)

    if out:
        return out

    for el in soup.find_all(True):
        classes = " ".join(el.get("class", [])).lower()
        attrs = " ".join(f"{k}={v}" for k, v in getattr(el, "attrs", {}).items()).lower()
        txt = _normalize_space(el.get_text(" ", strip=True))
        if not looks_like_title_text(txt):
            continue
        if not any(k in classes or k in attrs for k in ["movie", "title", "heading", "name"]):
            continue
        if not _container_has_showtime_or_runtime(el):
            continue
        key = (txt.lower(), getattr(el, "sourceline", None))
        if key not in seen:
            out.append(el)
            seen.add(key)
    return out

def _iter_block_tags(block: Tag, stop_tag: Optional[Tag]):
    if block.name in {"article", "section", "li", "div"}:
        for el in block.descendants:
            if isinstance(el, Tag):
                yield el
        return

    el = block.next_element
    while el is not None and el is not stop_tag:
        if isinstance(el, Tag):
            yield el
        el = el.next_element

def _extract_local_format_from_obj(obj: object) -> Optional[str]:
    if not isinstance(obj, dict):
        return None

    found: List[str] = []
    direct_keys = {"format", "formatlabel", "experience", "presentation", "label", "name"}
    container_keys = {"attributes", "attribute", "experiences", "presentations", "formats", "badges", "tags"}

    for k, v in obj.items():
        kl = str(k).lower()
        if kl in direct_keys or kl in container_keys:
            for s in _strings_one_level(v):
                label = normalize_format_label(s)
                if label and looks_like_format_label(label):
                    found.append(label)

    if not found:
        return None

    seen = set()
    uniq = []
    for s in found:
        low = s.lower()
        if low not in seen:
            uniq.append(s)
            seen.add(low)
    return " / ".join(uniq[:2])

def _extract_runtime_minutes_from_obj(obj: object) -> Optional[int]:
    candidates: List[int] = []

    def maybe_add(v: object) -> None:
        if isinstance(v, int) and 30 <= v <= 400:
            candidates.append(v)
        elif isinstance(v, str):
            m = AMC_RUNTIME_RE.search(v)
            if m:
                if m.group(1):
                    candidates.append(int(m.group(1)) * 60 + int(m.group(2)))
                elif m.group(3):
                    candidates.append(int(m.group(3)))

    def walk(x: object) -> None:
        if isinstance(x, dict):
            for k, v in x.items():
                kl = str(k).lower()
                if kl in {"runtime", "runtimeminutes", "runtimemin", "duration", "durationminutes", "featureruntime"}:
                    maybe_add(v)
                elif isinstance(v, (dict, list)):
                    walk(v)
        elif isinstance(x, list):
            for item in x:
                walk(item)

    walk(obj)
    return candidates[0] if candidates else None

def _extract_local_is_excluded_from_obj(obj: object) -> bool:
    if not isinstance(obj, dict):
        return False

    phrases = ("excluded from a-list", "not eligible for a-list")

    for k, v in obj.items():
        kl = str(k).lower()

        if any(tok in kl for tok in ("alist", "a-list", "excluded", "subscription", "eligible")):
            if isinstance(v, bool) and v:
                return True
            if isinstance(v, str) and any(p in v.lower() for p in phrases):
                return True

        if kl in {"attributes", "attribute", "badges", "labels", "tags"}:
            for s in _strings_one_level(v):
                sl = s.lower()
                if any(p in sl for p in phrases):
                    return True

    return False

def _iter_json_showtime_rows(obj: object, d: date, theatre_name: str, inherited_title: Optional[str] = None):
    wanted = d.isoformat()
    pacific = None
    try:
        from zoneinfo import ZoneInfo
        pacific = ZoneInfo("America/Los_Angeles")
    except Exception:
        pacific = None

    if isinstance(obj, dict):
        title = inherited_title
        for key in ("movieTitle", "movieName", "title", "name", "displayName"):
            val = obj.get(key)
            if isinstance(val, str) and looks_like_title_text(val):
                title = _normalize_space(val)
                break

        runtime_min = _extract_runtime_minutes_from_obj(obj)

        def time_from_value(v: object) -> Optional[Tuple[str, str]]:
            if not isinstance(v, str):
                return None
            s = v.strip()
            if not s:
                return None
            try:
                s2 = s.replace("Z", "+00:00")
                dt = datetime.fromisoformat(s2)
                if dt.tzinfo is not None and pacific is not None:
                    dt = dt.astimezone(pacific)
                date_txt = dt.date().isoformat()
                time_txt = dt.strftime("%I:%M %p").lstrip("0").lower()
                return date_txt, time_txt
            except Exception:
                pass
            m = re.search(r"(\d{4}-\d{2}-\d{2}).{0,3}(\d{1,2}:\d{2})", s)
            if m:
                hh, mm = m.group(2).split(":")
                hh_i = int(hh)
                suffix = "am" if hh_i < 12 else "pm"
                hh12 = hh_i % 12 or 12
                return m.group(1), f"{hh12}:{mm} {suffix}"
            return None

        time_keys = [
            "showDateTime", "showtimeDateTime", "dateTime", "startDate", "startDateTime",
            "startTime", "performanceDateTime", "scheduledDateTime", "businessDateTime"
        ]
        for k in time_keys:
            if k in obj:
                parsed = time_from_value(obj.get(k))
                if parsed and parsed[0] == wanted and title:
                    yield {
                        "movie_title": title,
                        "theatre": theatre_name,
                        "show_date": parsed[0],
                        "show_time": parsed[1],
                        "format_label": _extract_local_format_from_obj(obj),
                        "runtime_min": runtime_min,
                        "a_list_excluded": _extract_local_is_excluded_from_obj(obj),
                    }

        for v in obj.values():
            if isinstance(v, (dict, list)):
                yield from _iter_json_showtime_rows(v, d, theatre_name, title)

    elif isinstance(obj, list):
        for item in obj:
            yield from _iter_json_showtime_rows(item, d, theatre_name, inherited_title)

def extract_showtimes_from_json_scripts(html_txt: str, theatre_name: str, d: date) -> List[dict]:
    soup = BeautifulSoup(html_txt or "", "html.parser")
    out: List[dict] = []
    seen = set()
    wanted = d.isoformat()

    for script in soup.find_all("script"):
        txt = script.string or script.get_text(" ", strip=False) or ""
        if not txt or wanted not in txt:
            continue
        low = txt.lower()
        if "showtime" not in low and "showtimes" not in low and "datetime" not in low and "startdate" not in low:
            continue

        candidates: List[object] = []
        stype = (script.get("type") or "").lower()
        if stype == "application/ld+json":
            try:
                candidates.append(json.loads(txt))
            except Exception:
                pass
        if "__NEXT_DATA__" in txt or '"props"' in txt or '"pageProps"' in txt:
            try:
                candidates.append(json.loads(txt))
            except Exception:
                pass

        if not candidates:
            chunks = []
            if wanted in txt and (txt.lstrip().startswith("{") or txt.lstrip().startswith("[")):
                chunks.append(txt)
            for chunk in chunks:
                try:
                    candidates.append(json.loads(chunk))
                except Exception:
                    pass

        for cand in candidates:
            for row in _iter_json_showtime_rows(cand, d, theatre_name):
                key = (
                    row["movie_title"].strip().lower(),
                    row["theatre"].strip().lower(),
                    row["show_date"],
                    row["show_time"],
                    (row.get("format_label") or "").strip().lower(),
                )
                if row["show_date"] == wanted and key not in seen:
                    out.append(row)
                    seen.add(key)

    return out

def _collect_movie_blocks(soup: BeautifulSoup) -> List[Tuple[Tag, str]]:
    blocks: List[Tuple[Tag, str]] = []
    seen = set()

    # First preference: heading tags that contain a movie link.
    for el in soup.find_all(["h1", "h2", "h3", "h4", "h5"]):
        a = el.find("a", href=True)
        if not a or not href_matches_movie(a.get("href", "")):
            continue
        title = _normalize_space(a.get_text(" ", strip=True))
        key = (title.lower(), getattr(el, "sourceline", None))
        if title and key not in seen:
            blocks.append((el, title))
            seen.add(key)

    if blocks:
        return blocks

    # Second preference: title-like headings even if AMC removed the movie anchor wrapper.
    for el in _candidate_title_tags(soup):
        title = _normalize_space(el.get_text(" ", strip=True))
        key = (title.lower(), getattr(el, "sourceline", None))
        if title and key not in seen:
            blocks.append((el, title))
            seen.add(key)

    if blocks:
        return blocks

    # Fallback: any movie link, using a nearby semantic container if available.
    seen_titles = set()
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        if not href_matches_movie(href):
            continue
        title = _normalize_space(a.get_text(" ", strip=True))
        if not title:
            continue
        norm = re.sub(r"\s+", " ", title.strip().lower())
        if norm in seen_titles:
            continue
        blocks.append((_nearest_movie_container(a), title))
        seen_titles.add(norm)

    return blocks

    # Fallback: any movie link, using a nearby semantic container if available.
    seen_titles = set()
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        if not href_matches_movie(href):
            continue
        title = a.get_text(" ", strip=True)
        if not title:
            continue
        norm = re.sub(r"\s+", " ", title.strip().lower())
        if norm in seen_titles:
            continue
        blocks.append((_nearest_movie_container(a), title))
        seen_titles.add(norm)

    return blocks

def _likely_showtime_tag(el: Tag, txt: str) -> bool:
    if not txt or not TIME_RE.search(txt):
        return False
    clean = re.sub(r"\s+", " ", txt.strip())
    if len(clean) > 120:
        return False

    if el.name == "a" and href_matches_showtime(el.get("href", "")):
        return True

    classes = " ".join(el.get("class", [])).lower()
    attrs = " ".join(f"{k}={v}" for k, v in getattr(el, "attrs", {}).items()).lower()
    if el.name in {"button", "a"}:
        return True
    if any(k in classes for k in ["showtime", "session", "time"]):
        return True
    if any(k in attrs for k in ["showtime", "session", "time"]):
        return True

    # A very small element that is basically just a time label can still be a showtime pill.
    stripped = clean.lower()
    stripped = re.sub(r"up to \d+% off\.?","", stripped).strip()
    return bool(TIME_RE.fullmatch(stripped) or TIME_RE.match(stripped))


def _extract_local_format_near_tag(el: Tag) -> Optional[str]:
    candidates: List[str] = []

    def add_text(s: str) -> None:
        label = normalize_format_label(s)
        if label and looks_like_format_label(label):
            candidates.append(label)

    txt = _normalize_space(el.get_text(" ", strip=True))
    txt_wo_time = _normalize_space(TIME_RE.sub("", txt, count=1))
    if txt_wo_time:
        add_text(txt_wo_time)

    for attr_name in ("aria-label", "title", "data-format", "data-experience"):
        v = el.get(attr_name)
        if isinstance(v, str):
            add_text(v)

    parent = el.parent if isinstance(getattr(el, "parent", None), Tag) else None
    if parent is not None:
        for attr_name in ("aria-label", "title"):
            v = parent.get(attr_name)
            if isinstance(v, str):
                add_text(v)
        for sib in parent.find_all(recursive=False):
            if sib is el:
                continue
            add_text(sib.get_text(" ", strip=True))

    seen = set()
    for s in candidates:
        low = s.lower()
        if low not in seen:
            seen.add(low)
            return s
    return None

def is_a_list_excluded_near_tag(tag: Tag) -> bool:
    phrases = ("excluded from a-list", "not eligible for a-list")

    def has_phrase(text: str) -> bool:
        low = (text or "").lower()
        return any(p in low for p in phrases)

    txt = _normalize_space(tag.get_text(" ", strip=True))
    if has_phrase(txt):
        return True

    for attr_name, attr_val in getattr(tag, "attrs", {}).items():
        if isinstance(attr_val, str) and has_phrase(attr_val):
            return True
        if isinstance(attr_val, list) and any(isinstance(x, str) and has_phrase(x) for x in attr_val):
            return True

    parent = tag.parent if isinstance(getattr(tag, "parent", None), Tag) else None
    if parent is not None:
        ptxt = _normalize_space(parent.get_text(" ", strip=True))
        if has_phrase(ptxt):
            return True
        for attr_name, attr_val in getattr(parent, "attrs", {}).items():
            if isinstance(attr_val, str) and has_phrase(attr_val):
                return True
            if isinstance(attr_val, list) and any(isinstance(x, str) and has_phrase(x) for x in attr_val):
                return True
    return False

def scrape_amc_showtimes_for_date(session: requests.Session, theatre_name: str, showtimes_url: str, d: date) -> List[dict]:
    status, html_txt = fetch_amc_html(session, showtimes_url, params={"date": d.isoformat()})
    if status != 200:
        return []

    out: List[dict] = []
    seen = set()

    # Prefer JSON/script data first. It is usually more precise than walking the rendered DOM.
    json_rows = extract_showtimes_from_json_scripts(html_txt, theatre_name, d)
    for row in json_rows:
        dedupe_key = (
            row["movie_title"].strip().lower(),
            row["theatre"].strip().lower(),
            row["show_date"],
            row["show_time"],
            (row.get("format_label") or "").strip().lower(),
        )
        if dedupe_key not in seen:
            out.append(row)
            seen.add(dedupe_key)

    # DOM fallback only when JSON/script parsing found nothing.
    if not out:
        soup = BeautifulSoup(html_txt, "html.parser")
        movie_blocks = _collect_movie_blocks(soup)

        for idx, (block, title) in enumerate(movie_blocks):
            stop_tag = movie_blocks[idx + 1][0] if idx + 1 < len(movie_blocks) else None
            current_runtime = None

            for el in _iter_block_tags(block, stop_tag):
                txt = re.sub(r"\s+", " ", el.get_text(" ", strip=True))

                rt_m = AMC_RUNTIME_RE.search(txt)
                if rt_m and current_runtime is None:
                    if rt_m.group(1):
                        current_runtime = int(rt_m.group(1)) * 60 + int(rt_m.group(2))
                    else:
                        current_runtime = int(rt_m.group(3))

                if _likely_showtime_tag(el, txt):
                    time_m = TIME_RE.search(txt)
                    if time_m:
                        row = {
                            "movie_title": title,
                            "theatre": theatre_name,
                            "show_date": d.isoformat(),
                            "show_time": time_m.group(1).lower(),
                            "format_label": _extract_local_format_near_tag(el),
                            "runtime_min": current_runtime,
                            "a_list_excluded": is_a_list_excluded_near_tag(el),
                        }
                        dedupe_key = (
                            row["movie_title"].strip().lower(),
                            row["theatre"].strip().lower(),
                            row["show_date"],
                            row["show_time"],
                            (row["format_label"] or "").strip().lower(),
                        )
                        if dedupe_key not in seen:
                            out.append(row)
                            seen.add(dedupe_key)

    out = [r for r in out if r.get("show_date") == d.isoformat()]

    if not out:
        debug_dir = Path("build/amc_debug")
        debug_dir.mkdir(parents=True, exist_ok=True)
        slug = re.sub(r"[^a-z0-9]+", "-", theatre_name.lower()).strip("-")
        debug_path = debug_dir / f"{slug}-{d.isoformat()}.html"
        debug_path.write_text(html_txt or "", encoding="utf-8")
        print(f"[INFO] Saved AMC debug HTML to {debug_path}")

    return out


# =========================
# TITLE NORMALIZATION
# =========================
def candidate_title_variants(title: str) -> List[str]:
    t = re.sub(r"\s+", " ", (title or "").strip())
    if not t:
        return []
    variants = [t]
    for pat in [
        r"\s+\d{1,3}(st|nd|rd|th)\s+anniversary\s*$",
        r"\s+\d{1,3}th\s+anniversary\s*$",
        r"\s+early\s+access(\s+event)?\s*$",
        r"\s+sneak\s+peek\s*$",
        r"\s+fan\s+event\s*$",
        r"\s+re-?release\s*$",
        r"\s+opening\s+night\s*$",
        r"\s+q&a\s*$",
        r"\s+double\s+feature\s*$",
        r"\s+live\s+event\s*$",
    ]:
        c = re.sub(pat, "", t, flags=re.I).strip(" -:")
        if c and c.lower() != t.lower():
            variants.append(c)

    extra = []
    for v in variants:
        c = re.sub(r":\s+the\s+imax\s+experience$", "", v, flags=re.I).strip(" -:")
        if c and c.lower() != v.lower():
            extra.append(c)
        c = re.sub(r"\s*\(.*?\)", "", v).strip(" -:")
        if c and c.lower() != v.lower():
            extra.append(c)
    variants.extend(extra)

    seen, out = set(), []
    for v in variants:
        v = re.sub(r"\s+", " ", v).strip()
        k = v.lower()
        if v and k not in seen:
            seen.add(k)
            out.append(v)
    return out


# =========================
# ROTTEN TOMATOES (SLUG-FIRST, ROBUST PARSE)
# =========================
def rt_slugify(title: str) -> str:
    s = (title or "").lower().strip()
    s = re.sub(r"['’]", "", s)          # drop apostrophes
    s = re.sub(r"[^a-z0-9]+", "_", s)   # non-alnum -> underscore
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def _int0_100(x: Optional[str]) -> Optional[int]:
    if not x:
        return None
    x = str(x).strip()
    if not re.fullmatch(r"\d{1,3}", x):
        return None
    n = int(x)
    if 0 <= n <= 100:
        return n
    return None

def rt_parse_scores(decoded_html: str, soup: BeautifulSoup) -> Tuple[Optional[int], Optional[int]]:
    """
    Try (1) component attributes (most reliable), (2) embedded JSON, (3) visible text.
    Returns (audience, critic).
    """
    audience = None
    critic = None

    # (1) Component attributes (RT often renders <score-board ... tomatometerscore=".." audiencescore="..">)
    attr_candidates = []
    for attr in ["tomatometerscore", "tomatometerScore", "tomatometerscoreallcritics", "tomatometerscoreall"]:
        tag = soup.find(attrs={attr: True})
        if tag:
            attr_candidates.append((attr, tag.get(attr)))

    for attr, val in attr_candidates:
        n = _int0_100(val)
        if n is not None:
            critic = n
            break

    aud_attr_candidates = []
    for attr in ["audiencescore", "audienceScore", "popcornmeterscore", "popcornmeterScore"]:
        tag = soup.find(attrs={attr: True})
        if tag:
            aud_attr_candidates.append((attr, tag.get(attr)))

    for attr, val in aud_attr_candidates:
        n = _int0_100(val)
        if n is not None:
            audience = n
            break

    if critic is not None and audience is not None:
        return audience, critic

    # (2) Embedded JSON patterns in the HTML
    if critic is None:
        for rx in [RT_TOMA_JSON_2, RT_TOMA_JSON_1]:
            m = rx.search(decoded_html)
            if m:
                critic = _int0_100(m.group(1))
                if critic is not None:
                    break

    if audience is None:
        for rx in [RT_AUD_JSON_4, RT_AUD_JSON_3, RT_AUD_JSON_2, RT_AUD_JSON_1]:
            m = rx.search(decoded_html)
            if m:
                audience = _int0_100(m.group(1))
                if audience is not None:
                    break

    if critic is not None and audience is not None:
        return audience, critic

    # (3) Visible text fallback
    full_text = soup.get_text(" ", strip=True)
    full_text = re.sub(r"\s+", " ", full_text)

    # try to anchor around H1
    h1 = soup.find("h1")
    tail = full_text
    if h1:
        anchor = h1.get_text(" ", strip=True)
        if anchor:
            pos = full_text.lower().find(anchor.lower())
            if pos != -1:
                tail = full_text[pos:pos + 50000]

    if critic is None:
        m = (RT_TOMA_TXT_1.search(tail) or RT_TOMA_TXT_2.search(tail) or
             RT_TOMA_TXT_1.search(full_text) or RT_TOMA_TXT_2.search(full_text))
        if m:
            critic = _int0_100(m.group(1))

    if audience is None:
        m = RT_AUD_TXT_1.search(tail)
        if m:
            audience = _int0_100(m.group(1))
        else:
            m = RT_AUD_TXT_2.search(tail)
            if m:
                audience = _int0_100(m.group(2))
            else:
                m = RT_AUD_TXT_1.search(full_text)
                if m:
                    audience = _int0_100(m.group(1))
                else:
                    m = RT_AUD_TXT_2.search(full_text)
                    if m:
                        audience = _int0_100(m.group(2))

    return audience, critic

def rt_get_scores(
    session: requests.Session,
    title: str,
    cache: Dict[str, Tuple[Optional[int], Optional[int], Optional[str]]],
    debug: bool = False) -> Tuple[Optional[int], Optional[int], Optional[str]]:

    key = (title or "").lower().strip()
    if key in cache:
        return cache[key]

    # Get relevant years based on current date (2026)
    current_year = date.today().year
    years_to_check = [current_year, current_year + 1, current_year - 1]

    for q in candidate_title_variants(title):
        base_slug = rt_slugify(q)
        if not base_slug:
            continue

        # Create a list of URL attempts: [slug_2026, slug_2027, slug_2025, slug]
        url_attempts = [f"{RT_BASE}/m/{base_slug}_{y}" for y in years_to_check]
        url_attempts.append(f"{RT_BASE}/m/{base_slug}")

        for rt_url in url_attempts:
            status, raw_html = fetch_html(session, rt_url, params=None, tries=2)

            if debug:
                print(f"[RT] Testing {rt_url} -> Status: {status}")

            if status != 200 or not raw_html:
                continue

            # Double-check we didn't land on a '404' or 'Page Not Found'
            # RT often returns a 200 but shows a "sorry" page
            lower_html = raw_html.lower()
            if "404 - page not found" in lower_html or "couldn't find the page you're looking for" in lower_html:
                continue

            decoded = html_lib.unescape(raw_html)
            soup = BeautifulSoup(decoded, "html.parser")

            # Validate the title on the page matches our intent to avoid
            # false positives (e.g., scraping "Wicked" (1998) for "Wicked" (2024))
            # We look for the <h1> tag or the title meta tag
            # page_title = soup.find("h1")
            # page_title_text = page_title.get_text(strip=True).lower() if page_title else ""

            # Use fuzzy matching or simple inclusion to verify
            # if base_slug.replace("_", " ") not in page_title_text:
            #     if debug:
            #         print(f"[RT] Skipping {rt_url}: Title mismatch (Page says: {page_title_text})")
            #     continue


            aud, crit = rt_parse_scores(decoded, soup)
            cache[key] = (aud, crit, rt_url)
            return cache[key]

    cache[key] = (None, None, None)
    return cache[key]


# =========================
# IMDB (rating + runtime + canonical URL)
# =========================
def imdb_rating_runtime_url(
    session: requests.Session,
    title: str,
    cache: Dict[str, Tuple[Optional[float], Optional[int], Optional[str]]]
) -> Tuple[Optional[float], Optional[int], Optional[str]]:
    return cache.get((title or '').strip(), (None, None, None))


# =========================
# SORTING RULE (YOUR SPEC)
#   0) RT audience
#   1) IMDb (if no RT audience)
#   2) RT critic (if neither)
# =========================
def pick_primary(rt_aud: Optional[int], imdb: Optional[float], rt_crit: Optional[int]) -> Tuple[int, Optional[float], str]:
    """
    Returns (rank, score, source)
    rank: 0 audience, 1 imdb, 2 critic, 3 none
    score: audience / imdb*10 / critic
    """
    if rt_aud is not None:
        return 0, float(rt_aud), "RT_AUDIENCE"
    if imdb is not None:
        return 1, float(imdb) * 10.0, "IMDB"
    if rt_crit is not None:
        return 2, float(rt_crit), "RT_CRITIC"
    return 3, None, "NONE"


# =========================
# OUTPUT FORMATTING
# =========================
def short_fmt(s: Optional[str]) -> Optional[str]:
    # Check if 's' is None, or if it's a float (like NaN)
    if not isinstance(s, str):
        return None

    s = re.sub(r"\s+", " ", s).strip()
    return s if len(s) <= 40 else s[:37] + "..."

def build_showtimes_cell(df_st: pd.DataFrame) -> str:
    lines = []
    for theatre in sorted(df_st["theatre"].unique()):
        df_th = df_st[df_st["theatre"] == theatre].copy()

        lines.append(f"{theatre}")
        for show_date in sorted(df_th["show_date"].unique()):
            df_d = df_th[df_th["show_date"] == show_date].copy()
            df_d["t_sort"] = df_d["show_time"].apply(parse_time_for_sort)
            df_d = df_d.sort_values("t_sort")

            times = []
            for _, r in df_d.iterrows():
                excl = "⛔" if bool(r["a_list_excluded"]) else ""
                fmt = short_fmt(r.get("format_label"))
                fmt_txt = f" [{fmt}]" if fmt else ""
                times.append(f"{r['show_time']}{excl}{fmt_txt}")

            lines.append(f"• {show_date}: " + ", ".join(times))

    return "\n".join(lines)

# =========================
# RUN
# =========================
sat, sun = upcoming_weekend_pacific()
dates = [sat, sun]

session = make_session()

# Scrape showtimes
showtimes: List[dict] = []
for th in THEATRES:
    for d in dates:
        try:
            showtimes.extend(scrape_amc_showtimes_for_date(session, th["name"], th["url"], d))
            time.sleep(0.2)
        except Exception as e:
            print(f"[WARN] Failed scraping {th['name']} {d}: {e}")

if not showtimes:
    raise RuntimeError(
        "No fresh AMC showtimes were parsed for the requested dates. "
        "This run does not fall back to older data. "
        "AMC likely changed the page markup again or blocked the browser session."
    )

df_show = pd.DataFrame(showtimes)
# Add this line right after creating df_show:
df_show["format_label"] = df_show["format_label"].fillna("")

# Lookup scores (1 row per movie)
movie_titles = sorted(df_show["movie_title"].unique())
imdb_cache: Dict[str, Tuple[Optional[float], Optional[int], Optional[str]]] = build_imdb_lookup(session, movie_titles)
rt_cache: Dict[str, Tuple[Optional[int], Optional[int], Optional[str]]] = {}

movie_rows = []
for title in movie_titles:
    rt_aud, rt_crit, rt_url = rt_get_scores(session, title, rt_cache, debug=DEBUG_RT)
    imdb, imdb_runtime_min, imdb_url = imdb_rating_runtime_url(session, title, imdb_cache)

    rank, prim_score, prim_src = pick_primary(rt_aud, imdb, rt_crit)

    # Priority: AMC runtime first, IMDb runtime second
    amc_rt = df_show[df_show["movie_title"] == title]["runtime_min"].dropna()
    rt_min = int(amc_rt.iloc[0]) if not amc_rt.empty else imdb_runtime_min

    h, m = divmod(rt_min, 60) if rt_min else (0, 0)
    fmt_rt = f"{h}h {m}m" if h else (f"{m}m" if m else None)

    movie_rows.append({
        "movie_title": title,
        "runtime": fmt_rt,
        "runtime_min": rt_min,  # hidden helper
        "rt_audience": rt_aud,
        "imdb_rating": imdb,
        "rt_critic": rt_crit,
        "sort_rank": rank,
        "primary_score": prim_score,
        "primary_source": prim_src,
        "rt_url": rt_url,
        "imdb_url": imdb_url,
    })
    time.sleep(0.2)

df_movies = pd.DataFrame(movie_rows)

# Aggregate showtimes into one cell per movie + a_list_excluded_any (ANY showtime excluded)
agg = []
for title, df_st in df_show.groupby("movie_title"):
    agg.append({
        "movie_title": title,
        "a_list_excluded_any": bool(df_st["a_list_excluded"].fillna(False).any()),
        "showtimes": build_showtimes_cell(df_st),
    })
df_agg = pd.DataFrame(agg)

df_summary = (
    df_movies.merge(df_agg, on="movie_title", how="left")
            .sort_values(
                by=["sort_rank", "primary_score", "movie_title"],
                ascending=[True, False, True]
            )
            .reset_index(drop=True)
)

# Keep raw numeric columns and hidden URLs; postprocess_report.py will render score links.
df_display = df_summary.copy()


def fmt_imdb(x):
    if pd.isna(x):
        return ""
    try:
        return round(float(x), 1)
    except Exception:
        return _clean_missing_text(x)


df_display["imdb_rating"] = df_display["imdb_rating"].apply(fmt_imdb)

# Keep URL helper columns for the HTML postprocessor.
desired = [
    "movie_title",
    "runtime",
    "rt_critic",
    "rt_audience",
    "imdb_rating",
    "showtimes",
    "rt_url",
    "imdb_url",
]
df_display = df_display[[c for c in desired if c in df_display.columns]]

pd.set_option("display.max_colwidth", None)

print(f"Upcoming weekend (Pacific): {sat.isoformat()} (Sat), {sun.isoformat()} (Sun)")
html_out = df_display.to_html(index=False, escape=False).replace("\\n", "<br>")

css = """
<style>
table.dataframe th, table.dataframe td {
  text-align: left !important;
  vertical-align: top;
}
</style>
"""

display(HTML(css + html_out))



Upcoming weekend (Pacific): 2026-02-14 (Sat), 2026-02-15 (Sun)


movie_title,runtime,rt_audience,imdb_rating,rt_critic,showtimes
Stray Kids : The dominATE Experience,2h 26m,100.0,None,NaN,"AMC Orange 30• 2026-02-14: 10:35 pm [Laser at AMC]• 2026-02-15: 10:30 am [Laser at AMC], 2:15 pm [Laser at AMC], 6:00 pm [Laser at AMC], 9:45 pm [Laser at AMC]AMC Tustin 14 @ The District• 2026-02-15: 12:40 pm [Laser at AMC]"
The Rose: Come Back to Me,1h 37m,100.0,None,100.0,AMC Orange 30• 2026-02-15: 4:00 pm⛔ [Laser at AMC]
Time Hoppers: The Silk Road,1h 30m,99.0,None,NaN,AMC Orange 30• 2026-02-15: 8:45 am⛔ [Laser at AMC]
Melania,1h 44m,98.0,None,11.0,AMC Tustin 14 @ The District• 2026-02-15: 4:50 pm [Laser at AMC]
Nirvanna the Band the Show the Movie,1h 42m,98.0,None,97.0,"AMC Orange 30• 2026-02-14: 11:00 pm [Laser at AMC]• 2026-02-15: 11:00 am [Laser at AMC], 2:00 pm [Laser at AMC], 5:00 pm [Laser at AMC], 8:00 pm [Laser at AMC], 11:00 pm [Laser at AMC]AMC Tustin 14 @ The District• 2026-02-14: 10:05 pm [Laser at AMC]• 2026-02-15: 10:55 am [Laser at AMC], 1:40 pm [Laser at AMC], 4:30 pm [Laser at AMC], 7:20 pm [Laser at AMC], 10:10 pm [Laser at AMC]"
Sinners,2h 17m,96.0,None,97.0,AMC Orange 30• 2026-02-15: 11:45 am [Laser at AMC]
Solo Mio,1h 40m,96.0,None,80.0,"AMC Orange 30• 2026-02-14: 10:05 pm [Laser at AMC]• 2026-02-15: 10:00 am [Laser at AMC], 1:00 pm [Laser at AMC], 8:00 pm [Laser at AMC], 10:45 pm [Laser at AMC]AMC Tustin 14 @ The District• 2026-02-14: 10:10 pm [Laser at AMC]• 2026-02-15: 9:45 am [Laser at AMC], 4:20 pm [Laser at AMC], 8:05 pm [Laser at AMC]"
Zootopia 2,1h 47m,96.0,None,91.0,"AMC Orange 30• 2026-02-15: 10:05 am [Laser at AMC], 12:55 pm [Laser at AMC], 3:45 pm [Laser at AMC], 7:30 pm [Laser at AMC], 10:30 pm [Laser at AMC]AMC Tustin 14 @ The District• 2026-02-15: 10:40 am [Laser at AMC], 1:50 pm [Laser at AMC], 4:00 pm [Laser at AMC]AMC Woodbridge 5• 2026-02-15: 10:00 am [Laser at AMC], 12:45 pm [Laser at AMC], 3:30 pm [Laser at AMC], 6:15 pm [Laser at AMC]"
Goat,1h 39m,93.0,None,79.0,"AMC Orange 30• 2026-02-14: 10:45 pm [Laser at AMC]• 2026-02-15: 8:45 am [Laser at AMC], 9:45 am [Dolby Cinema at AMC], 10:30 am [Laser at AMC], 11:30 am [Laser at AMC], 12:30 pm [Dolby Cinema at AMC], 1:15 pm [Laser at AMC], 2:15 pm [Laser at AMC], 4:00 pm [Laser at AMC], 5:00 pm [Laser at AMC], 8:00 pm [Laser at AMC], 10:45 pm [Laser at AMC]AMC Tustin 14 @ The District• 2026-02-15: 10:00 am [Laser at AMC], 11:00 am [Laser at AMC], 1:00 pm [Dolby Cinema at AMC], 1:45 pm [Laser at AMC], 3:45 pm [Dolby Cinema at AMC], 5:00 pm [Laser at AMC], 6:15 pm [Laser at AMC], 9:00 pm [Laser at AMC]AMC Woodbridge 5• 2026-02-15: 10:00 am [Laser at AMC], 10:30 am [Laser at AMC], 11:00 am [Laser at AMC], 1:20 pm [Laser at AMC], 4:25 pm [Laser at AMC], 6:45 pm [Laser at AMC], 9:00 pm [Laser at AMC]"
Hamnet,2h 5m,93.0,None,86.0,"AMC Orange 30• 2026-02-15: 2:05 pm [Laser at AMC], 7:40 pm [Laser at AMC]"
